# Local ML Environment Check

This notebook verifies the Python libraries and CUDA GPU available to the RadGraph-XL project. It does not read restricted data.

In [ ]:
from __future__ import annotations

import importlib
import importlib.metadata
import json
import platform
import sys

PACKAGES = {
    "accelerate": "accelerate",
    "datasets": "datasets",
    "matplotlib": "matplotlib",
    "networkx": "networkx",
    "numpy": "numpy",
    "pandas": "pandas",
    "python-dotenv": "dotenv",
    "PyYAML": "yaml",
    "safetensors": "safetensors",
    "scikit-learn": "sklearn",
    "seaborn": "seaborn",
    "seqeval": "seqeval",
    "transformers": "transformers",
}

versions = {}
for package_name, import_name in PACKAGES.items():
    importlib.import_module(import_name)
    versions[package_name] = importlib.metadata.version(package_name)

print(json.dumps({
    "python": platform.python_version(),
    "executable": sys.executable,
    "packages": versions,
}, indent=2, sort_keys=True))

In [ ]:
import torch

cuda_report = {
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "devices": [],
}

for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    cuda_report["devices"].append({
        "index": index,
        "name": properties.name,
        "compute_capability": f"{properties.major}.{properties.minor}",
        "memory_gb": round(properties.total_memory / 1024**3, 2),
    })

print(json.dumps(cuda_report, indent=2))
assert cuda_report["cuda_available"], "CUDA is unavailable."
assert cuda_report["device_count"] >= 1, "No CUDA GPU was detected."

In [ ]:
from time import perf_counter

device = torch.device("cuda:0")
left = torch.randn((2048, 2048), device=device)
right = torch.randn((2048, 2048), device=device)

torch.cuda.synchronize()
started = perf_counter()
product = left @ right
torch.cuda.synchronize()
elapsed_ms = (perf_counter() - started) * 1000

compute_report = {
    "device": str(product.device),
    "shape": list(product.shape),
    "mean": float(product.mean().item()),
    "elapsed_ms": round(elapsed_ms, 3),
    "allocated_mb": round(torch.cuda.memory_allocated() / 1024**2, 2),
    "reserved_mb": round(torch.cuda.memory_reserved() / 1024**2, 2),
}

print(json.dumps(compute_report, indent=2))
assert product.is_cuda, "The matrix multiplication did not run on CUDA."